# MICrONS v7: can any of these brains make the abductive "Jump"?

A hidden-cause world (6 objects, 30 steps). One object secretly switches a light on when pushed, with 10% noise;
which one changes every episode and is never observable. Nothing happens unless the agent acts, so the cause can
only be found by experiment. The network predicts the outcome of pushing each object and pushes whichever it is
least certain about; weights are frozen at test time, so any inference lives in its activity.

Measured: how often it identifies the cause and how fast, **against the same network pushing at random** (does it
choose informative experiments?), whether the hidden cause is linearly decodable from its activity (did it form a
variable for something it never sees?), and transfer to two rules it never trained on - an **inverted** rule where
the cause switches the light OFF, and a two-cause rule.

Conditions: the sparse MICrONS brain (plastic), with local rewiring, complex random-phase, and a **GRU** baseline;
on real, random and scrambled wiring. 5 seeds.

In [ ]:
TASKS = "causal"
TASK_STEPS = "causal=3000"
SEEDS = 5
LR = 0.01
SAMPLE_PROB = 0.5
PLASTIC = True
MODES_FULL = "single,single+local,complex_randphase,gru"
MODES_NULL = "single,complex_randphase"

In [ ]:
import os, torch
os.makedirs("/kaggle/working/multiverse/data", exist_ok=True)
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("ready")

In [ ]:
%%writefile /kaggle/working/multiverse/connectome.py
from dataclasses import dataclass, replace

import h5py
import numpy as np
import pandas as pd

MICRONS_H5 = r"E:\microns_data\microns_mm3_connectome_v1181.h5"


@dataclass
class Connectome:
    name: str
    cell_type: np.ndarray
    inhibitory: np.ndarray
    pre: np.ndarray
    post: np.ndarray
    weight: np.ndarray
    input_types: tuple
    output_types: tuple
    xyz: np.ndarray | None = None

    @property
    def n(self):
        return len(self.cell_type)

    @property
    def input_idx(self):
        return np.flatnonzero(np.isin(self.cell_type, self.input_types))

    @property
    def output_idx(self):
        return np.flatnonzero(np.isin(self.cell_type, self.output_types))

    def shuffled(self, seed=0):
        """Degree-preserving null: every neuron keeps its in/out degree, partners are scrambled."""
        perm = np.random.default_rng(seed).permutation(len(self.post))
        return replace(self, name=f"{self.name}-shuffled", post=self.post[perm])

    def randomized(self, seed=0):
        """Erdos-Renyi null: same neuron and synapse counts, partners drawn uniformly (no connectome structure)."""
        rng = np.random.default_rng(seed)
        pre, post = rng.integers(0, self.n, len(self.pre)), rng.integers(0, self.n, len(self.post))
        keep = pre != post
        return replace(self, name=f"{self.name}-random", pre=pre[keep], post=post[keep], weight=self.weight[keep])

    def summary(self):
        types, counts = np.unique(self.cell_type, return_counts=True)
        mix = ", ".join(f"{t}:{c}" for t, c in zip(types, counts))
        return (f"{self.name}: {self.n} neurons, {len(self.pre)} connections, "
                f"{self.inhibitory.mean():.0%} inhibitory, input {len(self.input_idx)} / readout "
                f"{len(self.output_idx)} neurons [{mix}]")


def load_microns(path=MICRONS_H5, proofread="axon"):
    """Proofread neurons from MICrONS minnie65 v1181 and the condensed connections among them."""
    vp = pd.read_hdf(path, "/connectivity/condensed/vertex_properties")
    column = "status_axon" if proofread == "axon" else "status_dendrite"
    keep = (vp[column] == "t").to_numpy()
    with h5py.File(path, "r") as f:
        ij = f["connectivity/condensed/edge_indices/block0_values"][:]
        items = [x.decode() for x in f["connectivity/condensed/edges/block0_items"][:]]
        size = f["connectivity/condensed/edges/block0_values"][:, items.index("total_size")]
    new_id = np.full(len(vp), -1)
    new_id[keep] = np.arange(keep.sum())
    m = keep[ij[:, 0]] & keep[ij[:, 1]] & (ij[:, 0] != ij[:, 1])
    sub = vp[keep]
    return Connectome(
        name=f"microns-v1181-{proofread}",
        cell_type=sub["cell_type"].to_numpy(dtype=str),
        inhibitory=(sub["classification_system"] == "inhibitory_neuron").to_numpy(),
        pre=new_id[ij[m, 0]],
        post=new_id[ij[m, 1]],
        weight=size[m].astype(np.float64),
        input_types=("L4a", "L4b", "L4c"),
        output_types=("L5ET", "L5NP", "L5a", "L5b"),
        xyz=sub[["x_nm", "y_nm", "z_nm"]].to_numpy(dtype=float),
    )


def save_npz(conn, path):
    np.savez_compressed(
        path, name=conn.name, cell_type=conn.cell_type, inhibitory=conn.inhibitory,
        pre=conn.pre.astype(np.int32), post=conn.post.astype(np.int32), weight=conn.weight.astype(np.float32),
        input_types=np.array(conn.input_types), output_types=np.array(conn.output_types),
        xyz=conn.xyz.astype(np.float32),
    )


def load_npz(path):
    z = np.load(path)
    return Connectome(
        name=str(z["name"]), cell_type=z["cell_type"], inhibitory=z["inhibitory"],
        pre=z["pre"].astype(np.int64), post=z["post"].astype(np.int64), weight=z["weight"].astype(np.float64),
        input_types=tuple(z["input_types"].tolist()), output_types=tuple(z["output_types"].tolist()), xyz=z["xyz"],
    )


def synthetic(n=400, k_out=40, seed=0):
    """Stand-in with MICrONS-like statistics: laminar types, ~15% inhibition, distance-dependent, lognormal weights."""
    rng = np.random.default_rng(seed)
    xyz = rng.uniform(0, 1, (n, 3))
    depth = xyz[:, 1]
    excit = np.select([depth < 0.3, depth < 0.45, depth < 0.7], ["L23", "L4", "L5"], "L6")
    inhib = np.where(rng.uniform(size=n) < 0.6, "PTC", "DTC")
    inhibitory = rng.uniform(size=n) < 0.15
    cell_type = np.where(inhibitory, inhib, excit)

    d2 = ((xyz[:, None, :] - xyz[None, :, :]) ** 2).sum(-1)
    prob = np.exp(-d2 / (2 * 0.2 ** 2))
    np.fill_diagonal(prob, 0)
    prob = np.clip(prob / prob.sum(1, keepdims=True) * k_out, 0, 1)
    pre, post = np.nonzero(rng.uniform(size=(n, n)) < prob)
    weight = rng.lognormal(0.0, 1.0, len(pre))
    return Connectome("synthetic", cell_type, inhibitory, pre, post, weight, ("L4",), ("L5",), xyz)

In [ ]:
%%writefile /kaggle/working/multiverse/model.py
import math
import warnings

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta")

MODES = ("single", "separate", "separate_matched", "classical", "product", "quantum")
REWIRE_RULES = ("none", "random", "like", "local", "like_local")
DIAGONAL_MODES = ("separate", "separate_matched")
N_TYPE_PAIRS = 4  # (E,I) x (E,I) for pre/post


def _unitary(re, im):
    A = torch.complex(re, im)
    return torch.linalg.matrix_exp(A - A.conj().transpose(-2, -1))


def _diagonalizing_reflection(K):
    """Householder reflection sending the uniform product state to sum_a |a,a>/sqrt(K)."""
    v = torch.full((K * K,), 1.0 / K)
    t = torch.zeros(K * K)
    t[torch.arange(K) * (K + 1)] = 1.0 / math.sqrt(K)
    w = v - t
    return torch.eye(K * K) - 2 * torch.outer(w, w) / (w @ w)


def _spectral_radius(pre, post, values, n):
    M = sp.csr_matrix((values, (post, pre)), shape=(n, n))
    ev = spla.eigs(M, k=1, which="LM", return_eigenvectors=False, maxiter=10000, tol=1e-3)
    return float(np.abs(ev[0]))


def _faulty(w, faults):
    """Hardware-style faults on effective synapse weights, drawn reproducibly from faults['seed']."""
    if not faults:
        return w
    g = torch.Generator().manual_seed(faults.get("seed", 0))
    if faults.get("weight_noise"):
        w = w * (1 + faults["weight_noise"] * torch.randn(w.shape, generator=g).to(w.device))
    if faults.get("dead_frac"):
        w = w * (torch.rand(w.shape, generator=g) >= faults["dead_frac"]).to(w.device, w.dtype)
    if faults.get("bits"):
        levels = 2 ** faults["bits"] - 1
        top = w.abs().max()
        w = torch.sign(w) * torch.round(w.abs() / top * levels) / levels * top
    return w


def _crow(rows, n):
    crow = torch.zeros(n + 1, dtype=torch.long, device=rows.device)
    crow[1:] = torch.cumsum(torch.bincount(rows, minlength=n), 0)
    return crow


class _Pattern:
    """CSR sparsity pattern of an n x n coupling (duplicates merged) plus its transpose."""

    def __init__(self, dst, src, n):
        uniq, self.inverse = torch.unique(dst * n + src, return_inverse=True)
        self.n = n
        self.rows, self.cols = uniq // n, uniq % n
        self.crow = _crow(self.rows, n)
        self.t_order = torch.argsort(self.cols * n + self.rows)
        self.t_crow = _crow(self.cols, n)
        self.t_cols = self.rows[self.t_order]


class _MeshMul(torch.autograd.Function):
    """Recurrent input J x. vals[e, s] links brain a[e, s] of pre[e] to brain b[e, s] of post[e].
    Forward and dL/dx run as CSR multiplies (torch's sparse COO multiply is ~40x slower on CPU);
    dL/dvals is one brain-by-brain outer product per synapse."""

    @staticmethod
    def forward(ctx, vals, x, a, b, pre, post, pat, K):
        data = vals.new_zeros(len(pat.rows)).index_add_(0, pat.inverse, vals.reshape(-1))
        ctx.save_for_backward(data, x, a, b, pre, post)
        ctx.pat, ctx.K = pat, K
        A = torch.sparse_csr_tensor(pat.crow, pat.cols, data, (pat.n, pat.n))
        return (A @ x.T.contiguous()).T

    @staticmethod
    def backward(ctx, gy):
        data, x, a, b, pre, post = ctx.saved_tensors
        pat, K = ctx.pat, ctx.K
        g_vals = g_x = None
        if ctx.needs_input_grad[1]:
            At = torch.sparse_csr_tensor(pat.t_crow, pat.t_cols, data[pat.t_order], (pat.n, pat.n))
            g_x = (At @ gy.T.contiguous()).T
        if ctx.needs_input_grad[0]:
            B = x.shape[0]
            X = x.reshape(B, K, -1)[:, :, pre].permute(1, 2, 0).contiguous()  # (K, E, B)
            G = gy.reshape(B, K, -1)[:, :, post].permute(1, 2, 0).contiguous()
            edge = torch.arange(len(pre), device=pre.device)[:, None]
            if a.shape[1] == K * K:
                # all brain pairs: K large elementwise products beat E tiny matmuls
                outer = torch.stack([(X[i] * G).sum(-1) for i in range(K)])  # (K, K, E)
                g_vals = outer[a, b, edge]
            else:
                g_vals = (X[a, edge] * G[b, edge]).sum(-1)
        return g_vals, g_x, None, None, None, None, None, None


def _local_sigma(model, value):
    return float(value) if value else (model.xyz[model.pre] - model.xyz[model.post]).norm(dim=1).median().item()


@torch.no_grad()
def _activity_corr(model, u):
    model.forward(u, record=True)
    act = model.last_activity - model.last_activity.mean(0)
    norm = act.norm(dim=0).clamp_min(1e-8)
    return ((act.T @ act) / (norm[:, None] * norm[None, :])).reshape(-1)


@torch.no_grad()
def _rewire(model, step, total_steps, u, opt):
    """Structural plasticity for either brain type: every `rewire_every` steps (until 80% of training) prune the
    weakest synapses and grow as many new ones. Growth rules: random pairs, most correlated pairs (like), pairs sampled
    by distance (local), or correlation penalised by distance (like_local). New synapse signs follow the pre neuron."""
    if step % model.rewire_every or step >= 0.8 * total_steps:
        return
    N, E, dev = model.N, model.E, model.pre.device
    n_swap = int(model.rewire_frac * E)
    strength = (model.w_edge * torch.exp(model.log_w)).abs()
    slots = torch.topk(strength, n_swap, largest=False).indices
    keep = torch.ones(E, dtype=torch.bool, device=dev)
    keep[slots] = False

    corr = _activity_corr(model, u)
    locality = -((model.xyz[:, None, :] - model.xyz[None, :, :]) ** 2).sum(-1).reshape(-1) / (2 * model.local_sigma ** 2)
    if model.rewire == "random":
        score = torch.rand(N * N, device=dev)
    elif model.rewire == "like":
        score = corr.clone()
    elif model.rewire == "local":
        # Gumbel top-k: samples pairs with probability proportional to exp(locality)
        score = locality - torch.log(-torch.log(torch.rand(N * N, device=dev).clamp_min(1e-12)))
    else:
        score = corr + locality
    score[model.pre * N + model.post] = -float("inf")  # existing and just-pruned pairs
    score[torch.arange(N, device=dev) * (N + 1)] = -float("inf")
    new = torch.topk(score, n_swap).indices
    new_pre, new_post = new // N, new % N

    tau_new = model.cls[new_pre] * 2 + model.cls[new_post]
    kept_strength, kept_tau = strength[keep], model.tau[keep]
    typical = torch.stack([kept_strength[kept_tau == t].median() if (kept_tau == t).any() else kept_strength.median()
                           for t in range(N_TYPE_PAIRS)])
    model.pre[slots], model.post[slots], model.tau[slots] = new_pre, new_post, tau_new
    model.w_edge[slots] = typical[tau_new] * (1.0 - 2.0 * model.cls[new_pre].float())
    model.log_w[slots] = 0.0
    model.is_original[slots] = False
    state = opt.state.get(model.log_w, {})
    for key in ("exp_avg", "exp_avg_sq"):
        if key in state:
            state[key][slots] = 0.0
    model._after_rewire(slots)


@torch.no_grad()
def _wiring_stats(model):
    """Network statistics to compare against MICrONS: reciprocity, connection distance, degree spread, weight spread,
    and (when activity was recorded) how much more correlated connected neurons are than neuron pairs overall."""
    N = model.N
    keys = model.pre * N + model.post
    in_deg = torch.bincount(model.post, minlength=N).float()
    log_w = getattr(model, "log_w", None)
    w = (model.w_edge * torch.exp(log_w) if log_w is not None else model.w_edge).abs()
    s = {
        "reciprocity": torch.isin(model.post * N + model.pre, keys).float().mean().item(),
        "mean_conn_dist": (model.xyz[model.pre] - model.xyz[model.post]).norm(dim=1).mean().item(),
        "in_degree_cv": (in_deg.std() / in_deg.mean()).item(),
        "log_weight_sd": torch.log(w + 1e-12).std().item(),
    }
    if model.last_activity is not None:
        act = model.last_activity - model.last_activity.mean(0)
        norm = act.norm(dim=0).clamp_min(1e-8)
        corr = (act.T @ act) / (norm[:, None] * norm[None, :])
        off = ~torch.eye(N, dtype=torch.bool, device=corr.device)
        s["like_to_like_gap"] = (corr.reshape(-1)[keys].mean() - corr[off].mean()).item()
    return s


class Multiverse(nn.Module):
    """K copies of one connectome. Neurons hold existence amplitudes over brains; every synapse holds a
    two-neuron pair state (pre-brain, post-brain) produced by an entangling gate, and |amplitude|^2 decides
    which brain pairs that synapse links. Modes ablate each ingredient."""

    def __init__(self, conn, d_in, d_out, in_idx, out_idx, K=6, mode="quantum", radius=0.9, plastic=False,
                 rewire="none", rewire_every=100, rewire_frac=0.1, local_sigma=None):
        super().__init__()
        assert mode in MODES
        self.mode = mode
        self.K = K = 1 if mode == "single" else K
        self.N = N = conn.n

        # copies: rewiring edits these buffers in place and must not touch the shared connectome arrays
        pre = torch.tensor(conn.pre, dtype=torch.long)
        post = torch.tensor(conn.post, dtype=torch.long)
        inh = torch.tensor(conn.inhibitory)
        w = torch.as_tensor(conn.weight, dtype=torch.float32) * (1.0 - 2.0 * inh.float())[pre]
        w = w * (radius / _spectral_radius(conn.pre, conn.post, w.numpy(), N))
        cls = inh.long()
        tau = cls[pre] * 2 + cls[post]
        self.E = len(pre)
        self.register_buffer("pre", pre)
        self.register_buffer("post", post)
        self.register_buffer("w_edge", w)
        self.register_buffer("cls", cls)
        self.register_buffer("tau", tau)
        order = torch.argsort(tau)
        self.register_buffer("edge_order", order)
        self.register_buffer("edge_inv", torch.argsort(order))
        self.type_counts = torch.bincount(tau, minlength=N_TYPE_PAIRS).tolist()
        self.register_buffer("in_idx", torch.as_tensor(in_idx, dtype=torch.long))
        self.register_buffer("out_idx", torch.as_tensor(out_idx, dtype=torch.long))
        self.register_buffer("lesion", torch.ones(K, N))
        self._soft_pattern = None

        routes = torch.arange(K) * (K + 1) if mode in DIAGONAL_MODES else torch.arange(K * K)
        self.register_buffer("routes", routes)
        self.register_buffer("route_a", routes // K)
        self.register_buffer("route_b", routes % K)

        self.log_gain = nn.Parameter(torch.zeros(N_TYPE_PAIRS))
        self.plastic = plastic
        if plastic:
            # per-synapse log-strength; exp() keeps each synapse's excitatory/inhibitory sign
            self.log_w = nn.Parameter(torch.zeros(self.E))
        assert rewire in REWIRE_RULES
        assert rewire == "none" or (plastic and K == 1), "rewiring needs a single plastic brain"
        self.rewire, self.rewire_every, self.rewire_frac = rewire, rewire_every, rewire_frac
        self.register_buffer("is_original", torch.ones(self.E, dtype=torch.bool))
        self.register_buffer("xyz", torch.as_tensor(conn.xyz, dtype=torch.float32) / 1000)
        self.local_sigma = _local_sigma(self, local_sigma)
        self.last_activity = None
        self.faults = None
        self.init_stats = _wiring_stats(self)
        self.leak_logit = nn.Parameter(torch.zeros(2))
        self.bias = nn.Parameter(torch.zeros(N))
        self.inp = nn.Linear(d_in, len(in_idx), bias=False)
        self.readout = nn.Linear(len(out_idx), d_out)

        if mode == "separate_matched":
            # capacity control: the 2*K*N parameters the existence amplitudes use, spent on per-brain gains/biases
            self.brain_gain = nn.Parameter(torch.zeros(K, N))
            self.brain_bias = nn.Parameter(torch.zeros(K, N))

        if mode in ("classical", "product", "quantum"):
            self.psi_re = nn.Parameter(1.0 + 0.1 * torch.randn(N, K))
            self.psi_im = nn.Parameter(0.1 * torch.randn(N, K))
        if mode == "classical":
            init = torch.zeros(N_TYPE_PAIRS, K * K)
            init[:, torch.arange(K) * (K + 1)] = 4.0
            self.joint_logits = nn.Parameter(init)
        elif mode == "product":
            for name in ("pre_re", "pre_im", "post_re", "post_im"):
                setattr(self, name, nn.Parameter(0.01 * torch.randn(N_TYPE_PAIRS, K, K)))
        elif mode == "quantum":
            self.gate_re = nn.Parameter(0.01 * torch.randn(N_TYPE_PAIRS, K * K, K * K))
            self.gate_im = nn.Parameter(0.01 * torch.randn(N_TYPE_PAIRS, K * K, K * K))
            self.register_buffer("gate_base", _diagonalizing_reflection(K).to(torch.cfloat))

    def _psi(self):
        psi = torch.complex(self.psi_re, self.psi_im)
        return psi / torch.linalg.vector_norm(psi, dim=1, keepdim=True).clamp_min(1e-8)

    def existence(self):
        if self.mode in ("single",) + DIAGONAL_MODES:
            return self.w_edge.new_full((self.N, self.K), 1.0 / self.K)
        psi = self._psi()
        return psi.real ** 2 + psi.imag ** 2

    def _pair_amplitudes(self):
        K = self.K
        psi = self._psi()
        pair = (psi[self.pre].unsqueeze(2) * psi[self.post].unsqueeze(1)).reshape(-1, K * K)
        if self.mode == "product":
            U = _unitary(self.pre_re, self.pre_im)
            V = _unitary(self.post_re, self.post_im)
            G = torch.einsum("tab,tcd->tacbd", U, V).reshape(N_TYPE_PAIRS, K * K, K * K)
        else:
            G = _unitary(self.gate_re, self.gate_im) @ self.gate_base
        chunks = torch.split(pair[self.edge_order], self.type_counts)
        amp = torch.cat([c @ G[t].transpose(0, 1) for t, c in enumerate(chunks)])
        return amp[self.edge_inv]

    def pair_probs(self):
        """(E, R): probability that each synapse links route r = (pre-brain, post-brain)."""
        K, R = self.K, len(self.routes)
        if self.mode in ("single",) + DIAGONAL_MODES:
            return self.w_edge.new_full((self.E, R), 1.0 / R)
        if self.mode == "classical":
            p = self.existence()
            joint = p[self.pre].unsqueeze(2) * p[self.post].unsqueeze(1)
            q = F.softmax(self.joint_logits, -1).view(N_TYPE_PAIRS, K, K)[self.tau]
            P = (joint * q).reshape(self.E, K * K)
            # floor: existence and joint preferences can point at disjoint brains, underflowing a whole row to 0
            P = P + 1e-9
            return P / P.sum(1, keepdim=True)
        amp = self._pair_amplitudes()
        return amp.real ** 2 + amp.imag ** 2

    def coupling(self, sample=False):
        K, N = self.K, self.N
        P = self.pair_probs()
        w = self.w_edge * torch.exp(self.log_w) if self.plastic else self.w_edge
        scale = _faulty(torch.exp(self.log_gain)[self.tau] * w, self.faults).unsqueeze(1)
        # sqrt(K*P): 6 separate brains reproduce the original weights exactly
        amp = torch.sqrt(K * P + 1e-12)
        if sample:
            # measurement: each synapse is realized K times, routes drawn from P (energy-matched, K*E nonzeros)
            picks = torch.multinomial(P.detach(), K, replacement=True)
            q = amp.gather(1, picks)
            vals = scale * (1.0 + q - q.detach())
            a, b = self.route_a[picks], self.route_b[picks]
            pat = _Pattern((b * N + self.post[:, None]).reshape(-1), (a * N + self.pre[:, None]).reshape(-1), K * N)
        else:
            vals = scale * amp
            a, b = self.route_a.expand(self.E, -1), self.route_b.expand(self.E, -1)
            if self._soft_pattern is None or self._soft_pattern.rows.device != vals.device:
                self._soft_pattern = _Pattern((b * N + self.post[:, None]).reshape(-1),
                                              (a * N + self.pre[:, None]).reshape(-1), K * N)
            pat = self._soft_pattern
        return vals, a, b, pat

    def forward(self, u, sample=False, record=False, state=None, return_state=False):
        B, T, _ = u.shape
        K, N = self.K, self.N
        vals, a, b, pat = self.coupling(sample)
        p = self.existence()
        drive = torch.sqrt(K * p[self.in_idx]).T.unsqueeze(0)
        read = torch.sqrt(p[self.out_idx] / K).T.unsqueeze(0)
        leak = torch.sigmoid(self.leak_logit)[self.cls]
        x = u.new_zeros(B, K, N) if state is None else state
        outs, acts = [], []
        for t in range(T):
            inj = u.new_zeros(B, K, N).index_copy(2, self.in_idx, self.inp(u[:, t]).unsqueeze(1) * drive)
            rec = _MeshMul.apply(vals, x.reshape(B, K * N), a, b, self.pre, self.post, pat, K).reshape(B, K, N)
            pre = rec + inj
            if self.mode == "separate_matched":
                pre = torch.exp(self.brain_gain) * pre + self.brain_bias
            x = (1 - leak) * x + leak * torch.tanh(F.relu(pre + self.bias))
            x = x * self.lesion
            outs.append(self.readout((x[:, :, self.out_idx] * read).sum(1)))
            if record:
                acts.append(x[:, 0].detach())
        if record:
            self.last_activity = torch.cat(acts)
        out = torch.stack(outs, 1)
        return (out, x) if return_state else out

    @torch.no_grad()
    def set_lesion(self, neurons=None, brains=None):
        self.lesion.fill_(1.0)
        if neurons is not None:
            self.lesion[:, torch.as_tensor(neurons, device=self.lesion.device)] = 0.0
        if brains is not None:
            self.lesion[brains, :] = 0.0

    def maybe_rewire(self, step, total_steps, u, opt):
        _rewire(self, step, total_steps, u, opt)

    def _after_rewire(self, slots):
        order = torch.argsort(self.tau)
        self.edge_order.copy_(order)
        self.edge_inv.copy_(torch.argsort(order))
        self.type_counts = torch.bincount(self.tau, minlength=N_TYPE_PAIRS).tolist()
        self._soft_pattern = None

    def set_faults(self, **faults):
        self.faults = faults or None

    @torch.no_grad()
    def diagnostics(self):
        d = {"params": sum(p.numel() for p in self.parameters())}
        if self.plastic:
            d["synapse_log_change"] = self.log_w.abs().mean().item()
            d["synapses_changed_2x"] = (self.log_w.abs() > math.log(2)).float().mean().item()
        if self.rewire != "none":
            d["original_synapses_kept"] = self.is_original.float().mean().item()
        d.update({f"init_{k}": v for k, v in self.init_stats.items()})
        d.update(_wiring_stats(self))
        K = self.K
        if K == 1:
            return d
        P = self.w_edge.new_zeros(self.E, K * K)
        P[:, self.routes] = self.pair_probs()
        P = P.view(-1, K, K)
        pa, pb = P.sum(2, keepdim=True), P.sum(1, keepdim=True)
        mi = (P * (torch.log(P + 1e-12) - torch.log(pa * pb + 1e-12))).sum((1, 2))
        d["same_brain_frac"] = P.diagonal(dim1=1, dim2=2).sum(1).mean().item()
        d["brain_mi_bits"] = (mi.mean() / math.log(2)).item()
        p = self.existence()
        d["existence_entropy"] = (-(p * torch.log(p + 1e-12)).sum(1) / math.log(K)).mean().item()
        if self.mode in ("product", "quantum"):
            s2 = torch.linalg.svdvals(self._pair_amplitudes().view(-1, K, K)) ** 2
            d["entanglement_bits"] = (-(s2 * torch.log2(s2 + 1e-12)).sum(1)).mean().item()
        return d


BASELINE_MODES = ("gru",)


class GRUBrain(nn.Module):
    """Standard recurrent baseline with no connectome, same task interface (lesions, faults, diagnostics)."""

    def __init__(self, d_in, d_out, hidden=512):
        super().__init__()
        self.K, self.N, self.mode, self.rewire = 1, hidden, "gru", "none"
        self.cell = nn.GRUCell(d_in, hidden)
        self.readout = nn.Linear(hidden, d_out)
        self.register_buffer("lesion", torch.ones(1, hidden))
        self.faults = None
        self.last_activity = None

    def forward(self, u, sample=False, record=False, state=None, return_state=False):
        B, T, _ = u.shape
        w_hh = self.cell.weight_hh
        if self.faults:
            w_hh = _faulty(w_hh.reshape(-1), self.faults).reshape_as(w_hh)
        h = u.new_zeros(B, self.N) if state is None else state
        outs, acts = [], []
        for t in range(T):
            i_r, i_z, i_n = F.linear(u[:, t], self.cell.weight_ih, self.cell.bias_ih).chunk(3, 1)
            h_r, h_z, h_n = F.linear(h, w_hh, self.cell.bias_hh).chunk(3, 1)
            r, z = torch.sigmoid(i_r + h_r), torch.sigmoid(i_z + h_z)
            h = (1 - z) * torch.tanh(i_n + r * h_n) + z * h
            h = h * self.lesion
            outs.append(self.readout(h))
            if record:
                acts.append(h.detach())
        if record:
            self.last_activity = torch.cat(acts)
        out = torch.stack(outs, 1)
        return (out, h) if return_state else out

    @torch.no_grad()
    def set_lesion(self, neurons=None, brains=None):
        self.lesion.fill_(1.0)
        if neurons is not None:
            self.lesion[:, torch.as_tensor(neurons, device=self.lesion.device)] = 0.0

    def set_faults(self, **faults):
        self.faults = faults or None

    @torch.no_grad()
    def diagnostics(self):
        return {"params": sum(p.numel() for p in self.parameters())}


PHASE_MODES = ("complex_frozen", "complex_randphase", "complex_phase")


class PhaseBrain(nn.Module):
    """One connectome brain with complex-valued activity. Synapses carry phases in (-90, 90) degrees, so the real part
    keeps its excitatory/inhibitory sign while signals along different paths can interfere; readout sees Re z, Im z, |z|^2.
    complex_phase learns phases (strengths fixed); complex_frozen fixes phases at 0 and learns strengths instead,
    so both add exactly one parameter per synapse."""

    def __init__(self, conn, d_in, d_out, in_idx, out_idx, mode="complex_phase", radius=0.9,
                 rewire="none", rewire_every=100, rewire_frac=0.1, local_sigma=None):
        super().__init__()
        assert mode in PHASE_MODES
        self.mode, self.K, self.N, self.E = mode, 1, conn.n, len(conn.pre)
        N = conn.n
        # copies: rewiring edits these buffers in place and must not touch the shared connectome arrays
        pre = torch.tensor(conn.pre, dtype=torch.long)
        post = torch.tensor(conn.post, dtype=torch.long)
        inh = torch.tensor(conn.inhibitory)
        w = torch.as_tensor(conn.weight, dtype=torch.float32) * (1.0 - 2.0 * inh.float())[pre]
        w = w * (radius / _spectral_radius(conn.pre, conn.post, w.numpy(), N))
        cls = inh.long()
        for name, t in (("pre", pre), ("post", post), ("w_edge", w), ("cls", cls), ("tau", cls[pre] * 2 + cls[post])):
            self.register_buffer(name, t)
        self.register_buffer("in_idx", torch.as_tensor(in_idx, dtype=torch.long))
        self.register_buffer("out_idx", torch.as_tensor(out_idx, dtype=torch.long))
        self.register_buffer("lesion", torch.ones(1, N))
        self.register_buffer("route0", torch.zeros(self.E, 1, dtype=torch.long))
        self._pattern = None

        self.log_gain = nn.Parameter(torch.zeros(N_TYPE_PAIRS))
        self.leak_logit = nn.Parameter(torch.zeros(2))
        self.bias = nn.Parameter(torch.zeros(N))
        self.inp = nn.Linear(d_in, len(in_idx), bias=False)
        # reads Re z, Im z and |z|^2: |z|^2 alone is blind to the input's sign, since flipping it maps z -> -z
        self.readout = nn.Linear(3 * len(out_idx), d_out)
        if mode == "complex_phase":
            # all-zero phases keep every imaginary part at 0, where the |z|^2 readout gives phases exactly zero gradient
            self.phase_raw = nn.Parameter(0.4 * torch.randn(self.E))
        else:
            if mode == "complex_randphase":
                # same random draw as complex_phase's init but frozen: separates having phases from learning them
                self.register_buffer("phase_fixed", (math.pi / 2) * torch.tanh(0.4 * torch.randn(self.E)))
            self.log_w = nn.Parameter(torch.zeros(self.E))
        assert rewire in REWIRE_RULES
        assert rewire == "none" or mode != "complex_phase", "rewiring needs learnable synapse strengths"
        self.rewire, self.rewire_every, self.rewire_frac = rewire, rewire_every, rewire_frac
        self.register_buffer("is_original", torch.ones(self.E, dtype=torch.bool))
        self.register_buffer("xyz", torch.as_tensor(conn.xyz, dtype=torch.float32) / 1000)
        self.local_sigma = _local_sigma(self, local_sigma)
        self.last_activity = None
        self.faults = None
        self.init_stats = _wiring_stats(self)

    def phases(self):
        if self.mode == "complex_phase":
            return (math.pi / 2) * torch.tanh(self.phase_raw)
        if self.mode == "complex_randphase":
            return self.phase_fixed
        return self.w_edge.new_zeros(self.E)

    def forward(self, u, sample=False, record=False, state=None, return_state=False):
        B, T, _ = u.shape
        N = self.N
        w = self.w_edge * torch.exp(self.log_gain)[self.tau]
        if self.mode != "complex_phase":
            w = w * torch.exp(self.log_w)
        w = _faulty(w, self.faults)
        theta = self.phases()
        if self.faults and self.faults.get("phase_noise_deg"):
            g = torch.Generator().manual_seed(self.faults.get("seed", 0) + 1)
            theta = theta + math.radians(self.faults["phase_noise_deg"]) * torch.randn(self.E, generator=g).to(theta.device)
        j_re, j_im = (w * torch.cos(theta)).unsqueeze(1), (w * torch.sin(theta)).unsqueeze(1)
        if self._pattern is None or self._pattern.rows.device != w.device:
            self._pattern = _Pattern(self.post, self.pre, N)
        pat, r0 = self._pattern, self.route0
        leak = torch.sigmoid(self.leak_logit)[self.cls]
        z = u.new_zeros(2 * B, N) if state is None else state  # rows 0..B-1 real, rows B..2B-1 imaginary
        outs, acts = [], []
        for t in range(T):
            a = _MeshMul.apply(j_re, z, r0, r0, self.pre, self.post, pat, 1)
            b = _MeshMul.apply(j_im, z, r0, r0, self.pre, self.post, pat, 1)
            inj = u.new_zeros(B, N).index_copy(1, self.in_idx, self.inp(u[:, t]))
            v_re = a[:B] - b[B:] + inj
            v_im = b[:B] + a[B:]
            mag = torch.sqrt(v_re ** 2 + v_im ** 2 + 1e-8)
            scale = torch.tanh(F.relu(mag + self.bias)) / mag
            z = (1 - leak) * z + leak * torch.cat([v_re * scale, v_im * scale])
            z = z * self.lesion
            re, im = z[:B, self.out_idx], z[B:, self.out_idx]
            outs.append(self.readout(torch.cat([re, im, re ** 2 + im ** 2], 1)))
            if record:
                acts.append(torch.sqrt(z[:B] ** 2 + z[B:] ** 2).detach())
        if record:
            self.last_activity = torch.cat(acts)
        out = torch.stack(outs, 1)
        return (out, z) if return_state else out

    @torch.no_grad()
    def set_lesion(self, neurons=None, brains=None):
        self.lesion.fill_(1.0)
        if neurons is not None:
            self.lesion[:, torch.as_tensor(neurons, device=self.lesion.device)] = 0.0

    def maybe_rewire(self, step, total_steps, u, opt):
        _rewire(self, step, total_steps, u, opt)

    def _after_rewire(self, slots):
        self._pattern = None
        if self.mode == "complex_randphase":
            self.phase_fixed[slots] = (math.pi / 2) * torch.tanh(0.4 * torch.randn(len(slots), device=slots.device))

    def set_faults(self, **faults):
        self.faults = faults or None

    @torch.no_grad()
    def diagnostics(self):
        d = {"params": sum(p.numel() for p in self.parameters())}
        if self.mode == "complex_phase":
            deg = self.phases().abs() * 180 / math.pi
            d["mean_abs_phase_deg"] = deg.mean().item()
            d["synapses_phase_gt45"] = (deg > 45).float().mean().item()
        else:
            d["synapse_log_change"] = self.log_w.abs().mean().item()
            d["synapses_changed_2x"] = (self.log_w.abs() > math.log(2)).float().mean().item()
        if self.rewire != "none":
            d["original_synapses_kept"] = self.is_original.float().mean().item()
        d.update({f"init_{k}": v for k, v in self.init_stats.items()})
        d.update(_wiring_stats(self))
        return d

In [ ]:
%%writefile /kaggle/working/multiverse/tasks.py
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


def _device(model):
    return next(model.parameters()).device


@torch.no_grad()
def _diagnose(model, probe):
    model.forward(probe.to(_device(model)), record=True)
    return model.diagnostics()


HARDWARE_FAULTS = {
    "wnoise10": dict(weight_noise=0.1), "wnoise30": dict(weight_noise=0.3),
    "dead10": dict(dead_frac=0.1), "dead30": dict(dead_frac=0.3),
    "bits8": dict(bits=8), "bits4": dict(bits=4),
}
PHASE_FAULTS = {"phase10": dict(phase_noise_deg=10), "phase30": dict(phase_noise_deg=30)}


JOINT_LR_MULT = 10.0  # classical routing has 144 logits vs quantum's ~10k gate entries; at 1x they barely move


def fit(model, make_batch, loss_fn, steps, lr=3e-3, sample_prob=0.0):
    device = _device(model)
    named = list(model.named_parameters())
    groups = [{"params": [p for n, p in named if n != "joint_logits"], "lr": lr}]
    joint = [p for n, p in named if n == "joint_logits"]
    if joint:
        groups.append({"params": joint, "lr": lr * JOINT_LR_MULT})
    opt = torch.optim.AdamW(groups, weight_decay=0.0)
    model.train()
    t0 = time.time()
    for step in range(steps):
        u, y = make_batch()
        u, y = u.to(device), y.to(device)
        out = model(u, sample=bool(torch.rand(()) < sample_prob))
        loss = loss_fn(out, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if getattr(model, "rewire", "none") != "none":
            model.maybe_rewire(step + 1, steps, u, opt)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    model.eval()
    return (time.time() - t0) / max(steps, 1)


def memory(model_fn, steps, sample_prob, lr=3e-3,D=20, T=60, B=32):
    model = model_fn(d_in=1, d_out=D)

    def batch(bs=B):
        u = torch.rand(bs, T, 1) * 2 - 1
        y = torch.stack([torch.roll(u[..., 0], d, dims=1) for d in range(1, D + 1)], -1)
        return u, y

    sec = fit(model, batch, lambda o, y: F.mse_loss(o[:, D:], y[:, D:]), steps, lr=lr, sample_prob=sample_prob)
    u, y = batch(256)
    with torch.no_grad():
        o = model(u.to(_device(model)))[:, D:].reshape(-1, D).cpu()
    t = y[:, D:].reshape(-1, D)
    r2 = [torch.corrcoef(torch.stack([o[:, d], t[:, d]]))[0, 1].nan_to_num().item() ** 2 for d in range(D)]
    return {"capacity": sum(r2), "sec_per_step": sec, **model.diagnostics()}


def _narma10(bs, T):
    u = torch.rand(bs, T) * 0.5
    y = torch.zeros(bs, T)
    for t in range(9, T - 1):
        y[:, t + 1] = 0.3 * y[:, t] + 0.05 * y[:, t] * y[:, t - 9:t + 1].sum(1) + 1.5 * u[:, t - 9] * u[:, t] + 0.1
    # NARMA-10 occasionally diverges (~1% of batches reach inf); redraw those series
    bad = ~(torch.isfinite(y).all(1) & (y.abs().max(1).values < 1.5))
    if bad.any():
        u_new, y_new = _narma10(int(bad.sum()), T)
        u[bad], y[bad] = u_new[..., 0], y_new[..., 0]
    return u.unsqueeze(-1), y.unsqueeze(-1)


def narma10(model_fn, steps, sample_prob, lr=3e-3,T=100, washout=30, B=32):
    model = model_fn(d_in=1, d_out=1)
    loss = lambda o, y: F.mse_loss(o[:, washout:-1], y[:, washout + 1:])
    sec = fit(model, lambda: _narma10(B, T), loss, steps, lr=lr, sample_prob=sample_prob)
    u, y = _narma10(256, T)
    with torch.no_grad():
        o = model(u.to(_device(model))).cpu()
    target = y[:, washout + 1:]
    nrmse = torch.sqrt(F.mse_loss(o[:, washout:-1], target) / target.var()).item()
    return {"nrmse": nrmse, "sec_per_step": sec, **model.diagnostics()}


def _digits():
    d = load_digits()
    X = torch.tensor(d.images / 16.0, dtype=torch.float32)
    y = torch.tensor(d.target)
    perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(0))
    split = int(0.8 * len(y))
    return X[perm[:split]], y[perm[:split]], X[perm[split:]], y[perm[split:]]


def _as_sequence(img, blank=4):
    return torch.cat([img, img.new_zeros(img.shape[0], blank, 8)], 1)


@torch.no_grad()
def _probs(model, X, sample=False):
    return F.softmax(model(_as_sequence(X).to(_device(model)), sample=sample)[:, -1], -1).cpu()


def _acc(probs, y):
    return (probs.argmax(1) == y).float().mean().item()


def _ece(probs, y, bins=15):
    conf, pred = probs.max(1)
    correct = (pred == y).float()
    edges = torch.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.float().mean().item() * abs(conf[m].mean().item() - correct[m].mean().item())
    return total


def _entropy(p):
    return -(p * torch.log(p + 1e-12)).sum(1)


def digits(model_fn, steps, sample_prob, lr=3e-3,B=64, universes=8):
    Xtr, ytr, Xte, yte = _digits()
    model = model_fn(d_in=8, d_out=10)

    def batch():
        i = torch.randint(len(ytr), (B,))
        return _as_sequence(Xtr[i]), ytr[i]

    sec = fit(model, batch, lambda o, y: F.cross_entropy(o[:, -1], y), steps, lr=lr, sample_prob=sample_prob)
    res = {"sec_per_step": sec}
    probs = _probs(model, Xte)
    res["acc"] = _acc(probs, yte)
    res["ece"] = _ece(probs, yte)
    for sigma in (0.3, 0.6):
        res[f"acc_noise{sigma}"] = _acc(_probs(model, (Xte + sigma * torch.randn_like(Xte)).clamp(0, 1)), yte)

    ood = Xte.reshape(-1, 64).gather(1, torch.argsort(torch.rand(len(Xte), 64), 1)).reshape(-1, 8, 8)
    labels = [0] * len(Xte) + [1] * len(ood)
    soft_scores = torch.cat([_entropy(_probs(model, Xte)), _entropy(_probs(model, ood))])
    res["ood_auroc_soft"] = roc_auc_score(labels, soft_scores.numpy())

    def universe_entropy(X):
        return _entropy(torch.stack([_probs(model, X, sample=True) for _ in range(universes)]).mean(0))

    uni_scores = torch.cat([universe_entropy(Xte), universe_entropy(ood)])
    res["ood_auroc_universes"] = roc_auc_score(labels, uni_scores.numpy())
    res["acc_universes"] = _acc(torch.stack([_probs(model, Xte, sample=True) for _ in range(universes)]).mean(0), yte)

    for frac in (0.1, 0.3, 0.5):
        model.set_lesion(neurons=torch.randperm(model.N)[:int(frac * model.N)])
        res[f"acc_lesion{int(frac * 100)}"] = _acc(_probs(model, Xte), yte)
    if model.K > 1:
        model.set_lesion(brains=[0])
        res["acc_lose_one_brain"] = _acc(_probs(model, Xte), yte)
    model.set_lesion()
    faults = dict(HARDWARE_FAULTS, **(PHASE_FAULTS if hasattr(model, "phases") else {}))
    for name, f in faults.items():
        model.set_faults(**f)
        res[f"acc_{name}"] = _acc(_probs(model, Xte), yte)
    model.set_faults()
    res.update(_diagnose(model, _as_sequence(Xte[:256])))
    return res


def decision(model_fn, steps, sample_prob, lr=3e-3,T=30, B=64, cohs=(0.0, 0.05, 0.1, 0.2, 0.4)):
    model = model_fn(d_in=1, d_out=2)

    def batch(bs=B, coh=None):
        s = torch.randint(0, 2, (bs,))
        c = torch.tensor(cohs)[torch.randint(len(cohs), (bs,))] if coh is None else torch.full((bs,), coh)
        return (c * (2 * s - 1)).view(bs, 1, 1) + torch.randn(bs, T, 1), s

    sec = fit(model, batch, lambda o, y: F.cross_entropy(o[:, -1], y), steps, lr=lr, sample_prob=sample_prob)
    res = {"sec_per_step": sec}
    ratios = []
    for c in cohs:
        u, s = batch(1000, c)
        with torch.no_grad():
            acc = (model(u.to(_device(model)))[:, -1].argmax(1).cpu() == s).float().mean().item()
        res[f"acc_coh{c}"] = acc
        if c > 0:
            ratios.append(acc / (0.5 * (1 + math.erf(c * math.sqrt(T) / math.sqrt(2)))))
    res["frac_of_ideal"] = sum(ratios) / len(ratios)
    res.update(model.diagnostics())
    return res


def continual(model_fn, steps, sample_prob, lr=3e-3,B=64):
    Xtr, ytr, Xte, yte = _digits()
    model = model_fn(d_in=8, d_out=2)
    pairs = [(2 * k, 2 * k + 1) for k in range(5)]

    def subset(X, y, a, b):
        m = (y == a) | (y == b)
        return X[m], (y[m] == b).long()

    tests = [subset(Xte, yte, a, b) for a, b in pairs]
    history = []
    for a, b in pairs:
        X, Y = subset(Xtr, ytr, a, b)

        def batch():
            i = torch.randint(len(Y), (B,))
            return _as_sequence(X[i]), Y[i]

        fit(model, batch, lambda o, y: F.cross_entropy(o[:, -1], y), steps, lr=lr, sample_prob=sample_prob)
        history.append([_acc(_probs(model, Xt), Yt) for Xt, Yt in tests])
    final = history[-1]
    forgetting = [max(history[i][j] for i in range(j, len(pairs))) - final[j] for j in range(len(pairs) - 1)]
    res = {"final_avg_acc": sum(final) / len(final), "avg_forgetting": sum(forgetting) / len(forgetting)}
    res.update(_diagnose(model, _as_sequence(Xte[:128])))
    return res


def _state_features(state, B):
    """Flatten a model's recurrent state to (B, features) for the hidden-cause probe."""
    if state.dim() == 3:
        return state.reshape(B, -1)
    return torch.cat([state[:B], state[B:]], 1) if state.shape[0] == 2 * B else state


CAUSAL = {"K": 6, "T": 30, "lapse": 0.1}  # world size / episode length / noise, overridable from run.py


def _causal_rollout(model, B, K, T, n_causes, lapse, eps, policy, invert=False):
    """One batch of episodes. A hidden 'cause' object turns the light on when pushed (with lapse noise); the model
    predicts the outcome of pushing each object and pushes whichever it is least sure about."""
    device = _device(model)
    cause_idx = torch.rand(B, K, device=device).argsort(1)[:, :n_causes]
    cause = torch.zeros(B, K, dtype=torch.bool, device=device).scatter_(1, cause_idx, True)
    light = torch.zeros(B, device=device)
    prev = torch.full((B,), -1, dtype=torch.long, device=device)
    state, loss, preds, actions = None, 0.0, [], []
    for t in range(T):
        onehot = torch.zeros(B, K, device=device)
        seen = prev >= 0
        if seen.any():
            onehot[seen] = F.one_hot(prev[seen], K).float()
        obs = torch.cat([onehot, light[:, None], torch.full((B, 1), t / T, device=device)], 1)
        out, state = model(obs[:, None, :], state=state, return_state=True)
        logits = out[:, 0, :K]
        with torch.no_grad():
            a = (-(torch.sigmoid(logits) - 0.5).abs()).argmax(1) if policy == "model" \
                else torch.randint(K, (B,), device=device)
            if eps:
                rand = torch.randint(K, (B,), device=device)
                a = torch.where(torch.rand(B, device=device) < eps, rand, a)
            p_on = torch.where(cause.gather(1, a[:, None]).squeeze(1), 1 - lapse, lapse)
            light = torch.bernoulli(1 - p_on if invert else p_on)
        loss = loss + F.binary_cross_entropy_with_logits(logits.gather(1, a[:, None]).squeeze(1), light)
        preds.append(logits.argmin(1) if invert else logits.argmax(1))
        actions.append(a)
        prev = a
    return loss / T, torch.stack(preds, 1), torch.stack(actions, 1), cause, state


def _causal_metrics(preds, actions, cause, T):
    ok = cause.gather(1, preds).float()
    settled = torch.flip(torch.cummin(torch.flip(ok, [1]), 1).values, [1])  # 1 where correct from here on
    identified = settled.max(1).values > 0
    id_step = torch.where(identified, settled.argmax(1).float(), torch.full_like(ok[:, 0], float(T)))
    pushes = torch.tensor([len(set(actions[i, :int(id_step[i])].tolist())) for i in range(len(actions))],
                          dtype=torch.float32)
    return {"frac_identified": identified.float().mean().item(), "id_step": id_step.mean().item(),
            "distinct_pushes": pushes.mean().item()}


def causal(model_fn, steps, sample_prob, lr=3e-3, K=None, T=None, B=64, lapse=None, eps=0.1, eval_eps=512):
    """Abduction test: infer which hidden object causes the light by choosing informative experiments."""
    K, T, lapse = K or CAUSAL["K"], T or CAUSAL["T"], CAUSAL["lapse"] if lapse is None else lapse
    model = model_fn(d_in=K + 2, d_out=K)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    model.train()
    t0 = time.time()
    for _ in range(steps):
        loss = _causal_rollout(model, B, K, T, 1, lapse, eps, "model")[0]
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    res = {"sec_per_step": (time.time() - t0) / max(steps, 1)}
    with torch.no_grad():
        for policy in ("model", "random"):
            loss, preds, actions, cause, state = _causal_rollout(model, eval_eps, K, T, 1, lapse, 0.0, policy)
            tag = "" if policy == "model" else "_random"
            res[f"bce{tag}"] = loss.item()
            res.update({f"{k}{tag}": v for k, v in _causal_metrics(preds, actions, cause, T).items()})
            if policy == "model":
                feats, labels = _state_features(state, eval_eps).cpu().numpy(), cause.float().argmax(1).cpu().numpy()
        # rule-family transfer, never trained on: the cause now switches the light OFF instead of on
        _, preds, actions, cause_i, _ = _causal_rollout(model, eval_eps, K, T, 1, lapse, 0.0, "model", invert=True)
        res.update({f"inverted_{k}": v for k, v in _causal_metrics(preds, actions, cause_i, T).items()})
        # and a second rule: two objects work at once
        _, preds, actions, cause2, _ = _causal_rollout(model, eval_eps, K, T, 2, lapse, 0.0, "model")
        res.update({f"twocause_{k}": v for k, v in _causal_metrics(preds, actions, cause2, T).items()})
    half = eval_eps // 2
    probe = LogisticRegression(max_iter=1000).fit(feats[:half], labels[:half])
    res["probe_acc"] = float(probe.score(feats[half:], labels[half:]))
    res["probe_chance"] = 1.0 / K
    res.update(model.diagnostics())
    return res


NEURAL_DATA = None  # set by run.py: .npz of MICrONS movie frames + co-recorded neural traces


def _neural_dataset(path, n_neurons=1000, downsample=2):
    """MICrONS two-photon window: stimulus frames and z-scored traces, keeping the most reliable neurons."""
    z = np.load(path)
    stim = torch.tensor(np.asarray(z["stim"], dtype=np.float32)).unsqueeze(1)
    stim = F.avg_pool2d(stim, downsample).flatten(1)
    stim = (stim - stim.mean()) / (stim.std() + 1e-6)
    resp = torch.nan_to_num(torch.tensor(np.asarray(z["traces"], dtype=np.float32)))
    # lag-1 autocorrelation: real calcium responses are smooth in time, noise is not
    keep = (resp[1:] * resp[:-1]).mean(0).topk(min(n_neurons, resp.shape[1])).indices
    return stim, resp[:, keep]


def neural(model_fn, steps, sample_prob, lr=3e-3, T=40, B=32, n_neurons=1000):
    """Predict real MICrONS neural activity from the movie the mouse was watching."""
    stim, resp = _neural_dataset(NEURAL_DATA, n_neurons)
    n_train = int(0.8 * len(stim))
    model = model_fn(d_in=stim.shape[1], d_out=resp.shape[1])

    def batch():
        i = torch.randint(0, n_train - T, (B,))
        idx = i[:, None] + torch.arange(T)[None, :]
        return stim[idx], resp[idx]

    sec = fit(model, batch, lambda o, y: F.mse_loss(o[:, 5:], y[:, 5:]), steps, lr=lr, sample_prob=sample_prob)
    starts = torch.arange(n_train, len(stim) - T, T)
    windows = starts[:, None] + torch.arange(T)[None, :]
    with torch.no_grad():
        pred = model(stim[windows].to(_device(model))).cpu()
    p = pred[:, 5:].reshape(-1, resp.shape[1])
    y = resp[windows][:, 5:].reshape(-1, resp.shape[1])
    p, y = p - p.mean(0), y - y.mean(0)
    r = (p * y).sum(0) / (p.norm(dim=0) * y.norm(dim=0) + 1e-8)
    res = {"sec_per_step": sec, "mean_corr": r.mean().item(), "median_corr": r.median().item(),
           "frac_corr_gt0.2": (r > 0.2).float().mean().item(),
           "top100_corr": r.topk(min(100, len(r))).values.mean().item(), "n_neurons": float(len(r))}
    res.update(_diagnose(model, stim[windows[:8]]))
    return res


TRAIN_SPEEDS = torch.tensor([0.15, 0.3, 0.45, 0.6])
NOVEL_SPEEDS = torch.tensor([0.225, 0.375, 0.525])


def _oscillators(bs, T, speeds, noise=0.05):
    """Two 2D rotations whose angular speeds jump to new values at a random time: a small world whose rules change."""
    sw = torch.randint(25, 36, (bs,))
    i1 = torch.randint(len(speeds), (bs, 2))
    i2 = (i1 + torch.randint(1, len(speeds), (bs, 2))) % len(speeds)
    w1, w2 = speeds[i1][:, None, :], speeds[i2][:, None, :]
    t = torch.arange(T, dtype=torch.float32)[None, :, None]
    s = sw[:, None, None].float()
    phi = torch.rand(bs, 1, 2) * 2 * math.pi + torch.where(t < s, w1 * t, w1 * s + w2 * (t - s))
    clean = torch.cat([torch.cos(phi), torch.sin(phi)], -1)
    return clean + noise * torch.randn_like(clean), clean, sw


def _phase_errors(pred, clean, sw):
    """NRMSE of next-frame predictions before the rule change, in the 5 frames after it, and 10-19 frames after."""
    err = ((pred[:, :-1] - clean[:, 1:]) ** 2).mean(-1)  # err[:, t] is the error predicting frame t+1
    t = torch.arange(err.shape[1])[None, :]
    s = sw[:, None]
    windows = {"pre": (t >= 10) & (t < s - 1), "early": (t >= s - 1) & (t < s + 4), "late": (t >= s + 9) & (t < s + 19)}
    return {k: math.sqrt((err * m).sum().item() / m.sum().item() / 0.5) for k, m in windows.items()}


def worldmodel(model_fn, steps, sample_prob, lr=3e-3, T=60, B=32):
    model = model_fn(d_in=4, d_out=4)

    def batch():
        obs, clean, _ = _oscillators(B, T, TRAIN_SPEEDS)
        return obs, clean

    sec = fit(model, batch, lambda o, y: F.mse_loss(o[:, 5:-1], y[:, 6:]), steps, lr=lr, sample_prob=sample_prob)
    res = {"sec_per_step": sec}
    for name, speeds in (("seen", TRAIN_SPEEDS), ("novel", NOVEL_SPEEDS)):
        obs, clean, sw = _oscillators(512, T, speeds)
        with torch.no_grad():
            pred = model(obs.to(_device(model))).cpu()
        errs, persist = _phase_errors(pred, clean, sw), _phase_errors(obs, clean, sw)
        for k, v in errs.items():
            res[f"wm_{name}_{k}"] = v
        res[f"wm_{name}_recovery"] = errs["late"] / errs["pre"]
        res[f"wm_{name}_late_vs_persist"] = errs["late"] / persist["late"]
    res.update(_diagnose(model, obs[:64]))
    return res


TASKS = {"memory": memory, "narma10": narma10, "digits": digits, "decision": decision, "continual": continual,
         "worldmodel": worldmodel, "neural": neural, "causal": causal}

In [ ]:
%%writefile /kaggle/working/multiverse/run.py
import argparse
import json
import os
import time

import numpy as np
import torch

import tasks
from connectome import load_microns, load_npz, synthetic
from model import BASELINE_MODES, MODES, PHASE_MODES, REWIRE_RULES, GRUBrain, Multiverse, PhaseBrain
from tasks import TASKS

HERE = os.path.dirname(os.path.abspath(__file__))
DEFAULT_DATA = os.path.join(HERE, "data", "microns_v1181_axon.npz")


def build_connectome(args):
    if args.connectome == "synthetic":
        real = synthetic(n=args.n_neurons, seed=0)
    elif args.data.endswith(".npz"):
        real = load_npz(args.data)
    else:
        real = load_microns(args.data, args.proofread)
    conn = {"real": real, "shuffled": real.shuffled(seed=1), "random": real.randomized(seed=1)}[args.wiring]
    return real, conn


def make_model_fn(spec, conn, args, device, local_sigma):
    """A condition is 'mode' or 'mode+rewire_rule', e.g. 'single+like_local' or 'complex_randphase+random'."""
    mode, _, rule = spec.partition("+")
    rule = rule or "none"
    assert mode in MODES + PHASE_MODES + BASELINE_MODES and rule in REWIRE_RULES, spec
    if mode in BASELINE_MODES:
        return lambda d_in, d_out: GRUBrain(d_in, d_out, hidden=args.gru_hidden).to(device)
    common = dict(rewire=rule, rewire_every=args.rewire_every, rewire_frac=args.rewire_frac, local_sigma=local_sigma)
    if mode in PHASE_MODES:
        return lambda d_in, d_out: PhaseBrain(conn, d_in, d_out, conn.input_idx, conn.output_idx, mode,
                                              **common).to(device)
    return lambda d_in, d_out: Multiverse(conn, d_in, d_out, conn.input_idx, conn.output_idx, args.K, mode,
                                          plastic=args.plastic, **common).to(device)


def print_summary(results):
    print("\n=== summary (mean over seeds) ===")
    for task, by_mode in results.items():
        keys = [k for k in next(iter(by_mode.values()))[0]
                if all(k in r for runs in by_mode.values() for r in runs)]
        print(f"\n== {task} ==")
        print("condition".ljust(28) + "".join(k[:18].rjust(20) for k in keys))
        for mode, runs in by_mode.items():
            print(mode.ljust(28) + "".join(f"{np.mean([r[k] for r in runs]):20.4g}" for k in keys))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--connectome", choices=["microns", "synthetic"], default="microns")
    ap.add_argument("--data", default=DEFAULT_DATA, help=".npz subset, or the full MICrONS .h5")
    ap.add_argument("--proofread", choices=["axon", "dendrite"], default="axon", help=".h5 only")
    ap.add_argument("--n-neurons", type=int, default=400, help="synthetic only")
    ap.add_argument("--wiring", choices=["real", "shuffled", "random"], default="real")
    ap.add_argument("--neural-data", default="", help=".npz of MICrONS stimulus frames + neural traces")
    ap.add_argument("--modes", default=",".join(MODES), help="conditions: mode or mode+rewire_rule")
    ap.add_argument("--tasks", default=",".join(TASKS))
    ap.add_argument("--K", type=int, default=6)
    ap.add_argument("--steps", type=int, default=300)
    ap.add_argument("--task-steps", default="", help="per-task overrides, e.g. digits=1500,neural=2000")
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--sample-prob", type=float, default=0.5)
    ap.add_argument("--seeds", type=int, default=1)
    ap.add_argument("--plastic", action="store_true", help="learn a strength for every synapse (signs fixed)")
    ap.add_argument("--rewire-every", type=int, default=100)
    ap.add_argument("--rewire-frac", type=float, default=0.1)
    ap.add_argument("--gru-hidden", type=int, default=512)
    ap.add_argument("--causal-k", type=int, default=6, help="objects in the hidden-cause world")
    ap.add_argument("--causal-t", type=int, default=30, help="steps per abduction episode")
    ap.add_argument("--causal-lapse", type=float, default=0.1)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    tasks.NEURAL_DATA = args.neural_data
    tasks.CAUSAL = {"K": args.causal_k, "T": args.causal_t, "lapse": args.causal_lapse}
    task_steps = {k: int(v) for k, v in (item.split("=") for item in args.task_steps.split(",") if item)}
    device = torch.device(args.device)
    real, conn = build_connectome(args)
    # local growth uses the real connectome's typical connection length, whatever wiring we start from
    local_sigma = float(np.median(np.linalg.norm(real.xyz[real.pre] - real.xyz[real.post], axis=1))) / 1000
    print(f"{conn.summary()} | device={device} | local growth sigma {local_sigma:.3f}", flush=True)

    os.makedirs(os.path.join(HERE, "results"), exist_ok=True)
    tag = f"{conn.name}{'-plastic' if args.plastic else ''}_K{args.K}_{time.strftime('%Y%m%d-%H%M%S')}"
    path = os.path.join(HERE, "results", f"{tag}.json")

    results = {}
    # seed-first ordering keeps partial results balanced across conditions if a session is cut short
    for seed in range(args.seeds):
        for task in args.tasks.split(","):
            steps = task_steps.get(task, args.steps)
            for spec in args.modes.split(","):
                torch.manual_seed(seed)
                np.random.seed(seed)
                t0 = time.time()
                res = TASKS[task](make_model_fn(spec, conn, args, device, local_sigma), steps, args.sample_prob, lr=args.lr)
                res["steps"] = steps
                res["wall_sec"] = time.time() - t0
                results.setdefault(task, {}).setdefault(spec, []).append(res)
                print(f"[{task}] {spec} seed{seed}: " + ", ".join(f"{k}={v:.3g}" for k, v in res.items()), flush=True)
                with open(path, "w") as f:
                    json.dump({"args": vars(args), "connectome": conn.summary(), "results": results}, f, indent=1)

    print_summary(results)
    print(f"\nsaved {path}")


if __name__ == "__main__":
    main()

In [ ]:
import hashlib, os, subprocess, sys

H5 = "/tmp/microns_mm3_connectome_v1181.h5"
URL = "https://zenodo.org/api/records/13849415/files/microns_mm3_connectome_v1181.h5/content"
MD5 = "df6789abe8d379b594bf6b5c6fc0e0f8"
if not os.path.exists(H5):
    subprocess.run(["wget", "-q", "-O", H5, URL], check=True)
digest = hashlib.md5()
with open(H5, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        digest.update(chunk)
assert digest.hexdigest() == MD5, f"MD5 mismatch: {digest.hexdigest()}"
try:
    import tables
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tables"], check=True)

sys.path.insert(0, "/kaggle/working/multiverse")
from connectome import load_microns, load_npz, save_npz

save_npz(load_microns(H5), "/kaggle/working/multiverse/data/microns_v1181_axon.npz")
os.remove(H5)
print(load_npz("/kaggle/working/multiverse/data/microns_v1181_axon.npz").summary())

In [ ]:
import sys, time, torch
sys.path.insert(0, "/kaggle/working/multiverse")
import tasks
from connectome import load_npz
from model import GRUBrain, Multiverse, PhaseBrain

conn = load_npz("/kaggle/working/multiverse/data/microns_v1181_axon.npz")
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
K, T = 6, 30
builders = {
    "single plastic": lambda: Multiverse(conn, K + 2, K, conn.input_idx, conn.output_idx, 6, "single", plastic=True),
    "complex_randphase": lambda: PhaseBrain(conn, K + 2, K, conn.input_idx, conn.output_idx, "complex_randphase"),
    "gru": lambda: GRUBrain(K + 2, K),
}
for name, build in builders.items():
    m = build().to(dev)
    tasks._causal_rollout(m, 64, K, T, 1, 0.1, 0.1, "model")[0].backward()
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(3):
        tasks._causal_rollout(m, 64, K, T, 1, 0.1, 0.1, "model")[0].backward()
    if dev.type == "cuda":
        torch.cuda.synchronize()
    print(f"{name:18s} {(time.time() - t0) / 3:.3f} s per abduction training step ({T} env steps)")
del m
torch.cuda.empty_cache()

In [ ]:
import os, shlex, subprocess, sys, time, torch

base = [sys.executable, "-u", "/kaggle/working/multiverse/run.py", "--tasks", TASKS, "--task-steps", TASK_STEPS, "--seeds", str(SEEDS),
        "--lr", str(LR), "--sample-prob", str(SAMPLE_PROB), "--causal-k", "6", "--causal-t", "30"] + (["--plastic"] if PLASTIC else [])
plans = {
    "real": ("0", [["--wiring", "real", "--modes", MODES_FULL], ["--wiring", "random", "--modes", MODES_NULL]]),
    "shuffled": ("1" if torch.cuda.device_count() > 1 else "0", [["--wiring", "shuffled", "--modes", MODES_NULL]]),
}
procs = {}
for name, (gpu, steps_list) in plans.items():
    cmd = " && ".join(shlex.join(base + extra) for extra in steps_list)
    log = open(f"/kaggle/working/{name}.log", "w")
    procs[name] = subprocess.Popen(["bash", "-lc", cmd], stdout=log, stderr=subprocess.STDOUT,
                                   env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu})
    print(f"started {name} on GPU {gpu}: {len(steps_list)} wiring(s)")


def progress(name):
    done = [l for l in open(f"/kaggle/working/{name}.log").read().splitlines() if l.startswith("[") and " seed" in l]
    return f"{len(done)} runs, last: {done[-1][:120]}" if done else "0 runs"


t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"{(time.time() - t0) / 60:6.1f} min | " + " || ".join(f"{n}: {progress(n)}" for n in procs), flush=True)

for name, p in procs.items():
    print(f"\n===== {name} (exit code {p.returncode}) =====")
    print(open(f"/kaggle/working/{name}.log").read()[-12000:])

## Reading the results

- `frac_identified` / `id_step` vs `frac_identified_random` / `id_step_random`: the core test. Beating your own
  random-pushing policy means experiments are being *chosen*, not stumbled into.
- `probe_acc` vs `probe_chance` (1/6): is the hidden cause linearly readable from the recurrent state?
- `inverted_frac_identified`, `twocause_frac_identified`: transfer to rules never trained on. The inverted rule is
  the hard one.
- `distinct_pushes`: how many different objects it had to try.